# Visualization element examples

Explore Echoshader’s visualization elements using a processed NOAA Pacific hake survey dataset. The download cell retrieves and extracts `MVBS.zarr`, which is also used by the example application.

For examples using native RAW data, different vertical coordinates, and geometry matching, see [Echogram vertical coordinates and geometry](echogram_vertical_coordinates.ipynb).

## Import packages


In [1]:
import holoviews as hv
import numpy as np
import panel as pn
import xarray as xr

import echopype as ep
import echoshader
import geoviews as gv

from echopype.commongrid import resample_to_geometry
from echopype.qc import coerce_increasing_time, exist_reversed_time

pn.extension()
hv.extension("bokeh")
gv.extension("bokeh")

We define the ek500 colormap.

In [2]:
import matplotlib as mpl
import echopype.colormap

ek500_cmap = mpl.colormaps["ep.ek500"]

# Other visualization elements

The remaining examples use a processed Pacific hake survey dataset that includes `Sv`, `ping_time`, latitude, and longitude.


In [3]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

if not Path("MVBS.zarr/zarr.json").exists():
    urlretrieve(
        "https://github.com/echostack-org/echoshader/"
        "releases/download/example-data-v1/MVBS.zip",
        "MVBS.zip",
    )
    with ZipFile("MVBS.zip") as archive:
        archive.extractall(".")

survey_MVBS = xr.open_zarr("MVBS.zarr")
survey_MVBS

<xarray.Dataset> Size: 14MB
Dimensions:            (channel: 5, ping_time: 675, depth: 501)
Coordinates:
  * channel            (channel) StringDType() 80B 'WBT 400140-15 ES120-7C_ES...
  * ping_time          (ping_time) datetime64[ns] 5kB 2023-08-12T12:45:00 ......
  * depth              (depth) float64 4kB 0.0 1.0 2.0 3.0 ... 498.0 499.0 500.0
Data variables:
    frequency_nominal  (channel) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    latitude           (ping_time) float64 5kB dask.array<chunksize=(675,), meta=np.ndarray>
    longitude          (ping_time) float64 5kB dask.array<chunksize=(675,), meta=np.ndarray>
    Sv                 (channel, ping_time, depth) float64 14MB dask.array<chunksize=(5, 675, 501), meta=np.ndarray>
Attributes:
    processing_software_name:     echopype
    processing_software_version:  0.12.0
    processing_time:              2026-10-28T18:03:08+00:00
    processing_function:          commongrid.compute_MVBS
    processing_level:             Level 3A
    processing_level_url:         https://echopype.readthedocs.io/en/stable/p...

### Multi-frequency echograms

Pass multiple channels to create a vertical layout of echograms.
Customize their appearance using native HoloViews `.opts()`.

In [4]:
channels = (
    survey_MVBS.sortby("frequency_nominal").channel.values.tolist()
)

display_options = dict(
    cmap=ek500_cmap,
    clim=(-80, -30),
    colorbar=True,
    width=800,
    height=200,
)

survey_MVBS.eshader.echogram(
    channel=channels,
    vert_dim="depth",
).opts(
    hv.opts.Image(**display_options),
    hv.opts.QuadMesh(**display_options),
)

:Layout
   .Image.WBT_400141_hyphen_minus_15_ES18_ES                  :Image   [ping_time,depth]   (Sv)
   .Image.WBT_400143_hyphen_minus_15_ES38B_ES                 :Image   [ping_time,depth]   (Sv)
   .Image.WBT_400142_hyphen_minus_15_ES70_hyphen_minus_7C_ES  :Image   [ping_time,depth]   (Sv)
   .Image.WBT_400140_hyphen_minus_15_ES120_hyphen_minus_7C_ES :Image   [ping_time,depth]   (Sv)
   .Image.WBT_400145_hyphen_minus_15_ES200_hyphen_minus_7C_ES :Image   [ping_time,depth]   (Sv)

## Tricolor echogram

`tricolor_echogram()` combines three channels into RGB components. Here 120 kHz is mapped to red, 38 kHz to green, and 18 kHz to blue.


In [5]:
rgb_channels = [
    survey_MVBS.channel.sel(
        channel=survey_MVBS.frequency_nominal == frequency
    ).item()
    for frequency in (120000, 38000, 18000)
]

survey_MVBS.eshader.tricolor_echogram(
    channel=rgb_channels,
    vmin=-80,
    vmax=-30,
    vert_dim="depth",
).opts(
    width=800,
    height=450,
)


:RGB   [ping_time,depth]   (R,G,B)

## Track

`eshader.track()` plots the geographic position associated with each ping. A moving platform is shown as a path; stationary data are represented as a point.


In [6]:
survey_MVBS.eshader.track(
    tile="OpenTopoMap",
).opts(
    width=800,
    height=450,
)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Path.I   :Path   [Longitude,Latitude]   (Ping Time,Ping Index)
   .Points.I :Points   [Longitude,Latitude]   (Ping Time,Ping Index)

In [7]:
track = survey_MVBS.eshader.track(
    tile="OpenTopoMap",
).opts(
    gv.opts.Path(color="red", line_width=2),
).opts(
    width=800,
    height=450,
)

channel_18 = survey_MVBS.channel.sel(
    channel=survey_MVBS.frequency_nominal == 18000
).item()

eg = survey_MVBS.eshader.echogram(
    channel=channel_18,
    vert_dim="depth",
).opts(
    cmap=ek500_cmap,
    clim=(-80, -30),
    colorbar=True,
    width=800,
    height=450,
)

pn.Column(track, eg)

BokehModel(combine_events=True, render_bundle={'docs_json': {'0b12fcf3-4d70-41b1-bd6e-4ab4e5bfafb9': {'version…

## Histogram

`eshader.hist()` summarizes the `Sv` distribution by acoustic channel.


In [8]:
survey_MVBS.eshader.hist(
    bins=24,
    overlay=True,
).opts(
    width=900,
    height=450,
)


:NdOverlay   [channel]
   :Histogram   [Sv]   (Count)

## Statistics table

`eshader.table()` returns summary statistics for the complete dataset and individual channels.


In [9]:
survey_MVBS.eshader.table()


:Table   [index,Sum,WBT 400140-15 ES120-7C_ES,WBT 400141-15 ES18_ES,WBT 400142-15 ES70-7C_ES,WBT 400143-15 ES38B_ES,WBT 400145-15 ES200-7C_ES]

## Curtain

`eshader.curtain()` projects the acoustic observations along the vessel track. The Plotly backend provides an interactive 3-D view without requiring additional application state.


In [10]:
curtain_channel = survey_MVBS.channel.values[0]

survey_MVBS.eshader.curtain(
    channel=curtain_channel,
    ratio=0.001,
    engine="plotly",
    cmap="Viridis",
    clim=(-80, -30),
)

end of notebook

In [11]:
from datetime import datetime, timezone
import echopype
import echoshader
import xarray
import panel
import holoviews

print(f"echopype: {echopype.__version__}")
print(f"echoshader: {echoshader.__version__}")
print(f"xarray: {xarray.__version__}")
print(f"panel: {panel.__version__}")
print(f"holoviews: {holoviews.__version__}")
print(f"Executed: {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}")

echopype: 0.12.0
echoshader: 0.2.1
xarray: 2026.9.0
panel: 1.10.0
holoviews: 1.23.2
Executed: 2026-11-08 21:02 UTC
